In [ ]:
import os, torch, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()
!nvidia-smi

Sun May 31 05:06:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
pip install nbformat

In [ ]:
import nbformat

path = 'Pranav_LLM.ipynb'
with open(path, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

# Remove 'widgets' if 'state' is missing
if 'widgets' in nb['metadata'] and 'state' not in nb['metadata']['widgets']:
    del nb['metadata']['widgets']

with open(path, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)
print('Fixed notebook saved.')

FileNotFoundError: [Errno 2] No such file or directory: 'Pranav_LLM.ipynb'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip -q install -U transformers accelerate datasets peft trl bitsandbytes safetensors

In [ ]:
!ls -lh /content | head



In [ ]:
from datasets import load_dataset

# Update 'data_files' with the correct path to your train.jsonl file.
# For example, if it's in your Google Drive, it might look like this:
dataset = load_dataset("json", data_files="/content/drive/MyDrive/mini/content/train.jsonl", split="train") # <-- PLEASE UPDATE THIS LINE WITH YOUR ACTUAL PATH
# dataset = load_dataset("json", data_files="train.jsonl", split="train")
print("Dataset size:", len(dataset))
print(dataset[0])

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer
from datasets import load_dataset

# -----------------------------
# CONFIG
# -----------------------------
MODEL_NAME = "microsoft/phi-2"   # SAFE for Colab T4
DATA_PATH = "/content/drive/MyDrive/mini/content/train.jsonl"
OUT_DIR = "/content/car-assistant-qlora"

# -----------------------------
# LOAD DATASET
# -----------------------------
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print("Dataset size:", len(dataset))

# -----------------------------
# TOKENIZER
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 🔑 IMPORTANT: set sequence length HERE
tokenizer.model_max_length = 384

# -----------------------------
# QLoRA CONFIG
# -----------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# -----------------------------
# MODEL
# -----------------------------
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.config.use_cache = False

# -----------------------------
# LoRA CONFIG
# -----------------------------
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=1,

    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to="none"
)



def formatting_func(example):
    return example["text"]


trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=args,
    formatting_func=formatting_func,
    processing_class=tokenizer
)


# TRAIN
# -----------------------------
trainer.train()
trainer.save_model(OUT_DIR)

print("✅ Fine-tuning completed.")
print("✅ LoRA adapter saved at:", OUT_DIR)


In [ ]:
!ls -lh /content
!ls -lh /content/car-assistant-qlora | head


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "microsoft/phi-2"
OUT_DIR = "/content/car-assistant-qlora"   # make sure this folder exists

# 4-bit quant config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Base model
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
base.eval()

# Attach LoRA adapter
ft = PeftModel.from_pretrained(base, OUT_DIR)
ft.eval()

# Prompt
prompt = "<s>[INST] My car makes a clicking sound while starting. No warning lights. Provide possible causes and safe advice. [/INST]"

# ✅ Tokenize -> generate -> decode
inputs = tokenizer(prompt, return_tensors="pt").to(ft.device)

with torch.no_grad():
    output_ids = ft.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

text = tokenizer.decode(output_ids[0], skip_special_tokens=False)

# Optional: keep only assistant part
if "[/INST]" in text:
    text = text.split("[/INST]", 1)[-1].strip()

print(text.replace("</s>", "").strip())



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!zip -r car_assistant_adapter.zip /content/car-assistant-qlora


	zip warning: name not matched: /content/car-assistant-qlora

zip error: Nothing to do! (try: zip -r car_assistant_adapter.zip . -i /content/car-assistant-qlora)


In [ ]:
!pip -q install -U gradio transformers accelerate peft bitsandbytes safetensors


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!ls /content/drive/MyDrive/mini/content
!ls -lah /content/drive/MyDrive/mini/content/car-assistant-qlora


In [ ]:
import re

OBD_RE = re.compile(r"\bP[0-3][0-9A-F]{3}\b", re.IGNORECASE)

OBD_HINTS = {
    "P0300": "Random/Multiple Cylinder Misfire Detected",
    "P0301": "Cylinder 1 Misfire Detected",
    "P0171": "System Too Lean (Bank 1)",
    "P0420": "Catalyst System Efficiency Below Threshold (Bank 1)",
    "P0128": "Coolant Thermostat (Coolant Temp Below Regulating Temp)",
    "P0455": "EVAP System Large Leak Detected",
    "P0700": "Transmission Control System Malfunction",
}

def normalize_obd_codes(obd_text: str):
    return sorted(set(m.group(0).upper() for m in OBD_RE.finditer(obd_text or "")))

def explain_obd(codes):
    if not codes:
        return "None provided."
    lines = []
    for c in codes:
        lines.append(f"{c}: {OBD_HINTS.get(c, 'Unknown/General OBD-II code (lookup needed)')}")
    return "\n".join(lines)

print("✅ OBD helper ready.")


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "microsoft/phi-2"
ADAPTER_DIR = "/content/drive/MyDrive/mini/content/car-assistant-qlora"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
base_model.eval()
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

model.eval()

print("✅ LoRA adapter attached. Fine-tuned model ready.")

In [ ]:
import torch, re

REQUIRED_HEADINGS = [
    "Symptom Summary:",
    "OBD-II Interpretation:",
    "Likely Causes",
    "Risk Level",
    "Safe Checks",
    "Do NOT Do",
    "Next Action"
]

def looks_valid(text: str) -> bool:
    t = text.strip()
    return sum(h.lower() in t.lower() for h in REQUIRED_HEADINGS) >= 4  # at least 4 headings appear

# Ban common code-y starts (helps a LOT)
bad_words = ["def ", "class ", "import ", "argparse", "parser =", "raw_input", "print(", "super().__init__", "pydantic"]
bad_words_ids = [tokenizer(bw, add_special_tokens=False).input_ids for bw in bad_words if len(tokenizer(bw, add_special_tokens=False).input_ids) > 0]

SYSTEM_FORMAT = """Return output in this exact format (no code, no HTML):

Symptom Summary:
OBD-II Interpretation:
Likely Causes (ranked):
Risk Level (Low/Medium/High):
Safe Checks (user can do):
Do NOT Do:
Next Action:
"""

def generate_once(symptoms: str, obd_text: str, temperature: float):
    codes = normalize_obd_codes(obd_text or "")
    codes_str = ", ".join(codes) if codes else "None"
    obd_info = explain_obd(codes)

    prompt = (
        f"<s>[INST] You are a car diagnostic assistant. "
        f"IMPORTANT: Do NOT write code. Do NOT write HTML. "
        f"Only write the sections exactly as specified.\n\n"
        f"{SYSTEM_FORMAT}\n"
        f"Symptoms: {symptoms.strip()}\n"
        f"OBD Codes: {codes_str}\n"
        f"OBD Notes:\n{obd_info}\n"
        f"[/INST]\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.25,
            bad_words_ids=bad_words_ids,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(out_ids[0], skip_special_tokens=False)
    if "[/INST]" in text:
        text = text.split("[/INST]", 1)[-1]
    return text.replace("</s>", "").replace("<s>", "").strip()

def baseline_fallback(symptoms: str, obd_text: str):
    codes = normalize_obd_codes(obd_text or "")
    codes_str = ", ".join(codes) if codes else "None"

    # Expand OBD hints a bit
    local_obd = dict(OBD_HINTS)
    local_obd.update({
        "P0302": "Cylinder 2 Misfire Detected",
        "P0303": "Cylinder 3 Misfire Detected",
        "P0304": "Cylinder 4 Misfire Detected",
        "P0351": "Ignition Coil A Primary/Secondary Circuit",
        "P0352": "Ignition Coil B Primary/Secondary Circuit",
        "P0117": "Engine Coolant Temp Circuit Low Input",
        "P0118": "Engine Coolant Temp Circuit High Input",
    })

    def explain_obd2(codes):
        if not codes:
            return "None provided."
        return "\n".join([f"{c}: {local_obd.get(c, 'Unknown/General OBD-II code (lookup needed)')}" for c in codes])

    obd_info = explain_obd2(codes)

    s = (symptoms or "").lower()

    # Risk heuristic
    high_risk_terms = ["brake", "overheat", "steam", "smoke", "fuel smell", "petrol smell", "raw gas", "burning smell"]
    risk = "High" if any(t in s for t in high_risk_terms) else "Medium"

    # --- Rule-based likely causes ---
    causes = []
    safe_checks = []
    do_not = []
    next_action = []

    # Fuel smell + shaking -> possible misfire + fuel leak
    if ("fuel smell" in s or "petrol smell" in s or "raw gas" in s) and ("shake" in s or "shaking" in s or "rough" in s):
        causes = [
            "Misfire causing unburnt fuel (spark plug/ignition coil/injector)",
            "Fuel leak in engine bay or EVAP system leak",
            "Air-fuel imbalance (MAF/MAP sensor, vacuum leak)"
        ]
        safe_checks = [
            "If smell is strong, turn off engine and check for visible leaks under hood/near fuel lines (no flames).",
            "Check if Check Engine Light is flashing (flashing = stop driving).",
            "Note whether shaking happens at idle only or also while driving."
        ]
        do_not = [
            "Do not continue driving if fuel smell is strong or if the engine is shaking heavily.",
            "Do not smoke or use open flame near the vehicle."
        ]
        next_action = [
            "Tow to a service center if fuel smell is strong.",
            "Ask for misfire diagnosis (spark plugs, coils, injectors) and fuel leak inspection."
        ]
        risk = "High"

    # Clicking on start -> battery/starter
    elif ("click" in s or "clicking" in s) and ("start" in s or "starting" in s or "crank" in s):
        causes = [
            "Weak battery or loose/corroded terminals",
            "Starter relay/solenoid issue",
            "Starter motor issue"
        ]
        safe_checks = [
            "Check battery terminals for looseness/corrosion (engine OFF).",
            "Observe if headlights dim a lot during crank attempt.",
            "Try jump start if available (follow safe procedure)."
        ]
        do_not = [
            "Do not crank repeatedly for long durations.",
            "Do not ignore burning smell or smoke."
        ]
        next_action = [
            "Get battery + starter + charging system tested.",
            "If it starts with jump, battery/charging is likely."
        ]

    # Overheating -> cooling system
    elif ("overheat" in s or "temperature" in s or "temp" in s or "steam" in s):
        causes = [
            "Low coolant / coolant leak",
            "Thermostat stuck",
            "Radiator fan not working",
            "Water pump issue"
        ]
        safe_checks = [
            "Stop driving if gauge goes to red; let engine cool.",
            "Check coolant level only when engine is cool.",
            "Look for coolant leaks under the car."
        ]
        do_not = [
            "Do not open radiator cap when hot.",
            "Do not continue driving while overheating."
        ]
        next_action = [
            "Cooling system inspection recommended immediately.",
            "Tow if overheating persists."
        ]
        risk = "High"

    # Brakes -> braking system
    elif "brake" in s or "braking" in s or "grinding" in s or "soft pedal" in s:
        causes = [
            "Worn brake pads/rotors",
            "Brake fluid leak / air in brake lines",
            "Stuck caliper"
        ]
        safe_checks = [
            "Check brake fluid level if accessible.",
            "Test braking gently at low speed in a safe area."
        ]
        do_not = [
            "Do not drive at high speed.",
            "Do not ignore grinding noises."
        ]
        next_action = [
            "Brake inspection urgently recommended."
        ]
        risk = "High"

    # Misfire codes -> engine misfire bucket
    elif any(c.startswith("P03") for c in codes) or ("misfire" in s or "rough idle" in s or "shaking" in s):
        causes = [
            "Spark plug worn/fouled",
            "Ignition coil issue",
            "Fuel injector issue",
            "Vacuum leak / MAF sensor issue"
        ]
        safe_checks = [
            "If Check Engine Light is flashing, avoid driving.",
            "Note if issue worsens under acceleration.",
            "Check for loose intake hose if visible."
        ]
        do_not = [
            "Do not keep driving long distances with severe misfire."
        ]
        next_action = [
            "Request misfire diagnosis (plugs/coils/injectors) and read freeze-frame data."
        ]
        risk = "High" if "flashing" in s else risk

    # Default generic
    else:
        causes = [
            "General engine/electrical issue",
            "Sensor or maintenance-related fault"
        ]
        safe_checks = [
            "Note when the symptom occurs (startup/idle/acceleration).",
            "Provide vehicle model/year and any warning lights.",
            "Scan OBD-II codes if possible."
        ]
        do_not = [
            "Do not ignore persistent warning lights."
        ]
        next_action = [
            "If persistent, visit a technician with the symptom notes."
        ]

    # Build response in your required format
    causes_fmt = "\n".join([f"{i+1}) {c}" for i, c in enumerate(causes)])
    checks_fmt = "\n".join([f"- {c}" for c in safe_checks])
    donot_fmt = "\n".join([f"- {d}" for d in do_not])
    next_fmt = "\n".join([f"- {n}" for n in next_action])

    return f"""Symptom Summary:
- {symptoms.strip()}

OBD-II Interpretation:
- Codes: {codes_str}
{obd_info}

Likely Causes (ranked):
{causes_fmt}

Risk Level (Low/Medium/High):
- {risk}

Safe Checks (user can do):
{checks_fmt}

Do NOT Do:
{donot_fmt}

Next Action:
{next_fmt}
"""


def diagnose(symptoms: str, obd_text: str):
    symptoms = (symptoms or "").strip()
    if not symptoms:
        return "Please enter symptoms (e.g., clicking sound when starting, warning light, vibration, smell)."


    out1 = generate_once(symptoms, obd_text, temperature=0.5)
    if looks_valid(out1):
        return out1


    out2 = generate_once(symptoms, obd_text, temperature=0.2)
    if looks_valid(out2):
        return out2


    return baseline_fallback(symptoms, obd_text)

print("✅ Guardrailed diagnose() ready.")
print(diagnose("Clicking sound when starting, no warning lights", "P0300"))
print("\n" + "="*60 + "\n")
print(diagnose("Strong raw gas smell and heavy engine shaking", "P0302"))




In [ ]:
import gradio as gr


EXAMPLES = [
    ["Clicking sound when starting, no warning lights", "P0300"],
    ["Strong raw gas smell and heavy engine shaking", "P0302"],
    ["Engine overheating, temperature gauge goes high, steam near hood", "P0128"],
    ["Grinding noise while braking, brake pedal feels soft", ""],
]

with gr.Blocks(title="Car Assistant LLM — Diagnosis + OBD-II") as demo:
    gr.Markdown("# 🚗 Car Assistant LLM — Problem Diagnosis & Guidance")
    gr.Markdown(
        "Enter your car problem systems and  OBD-II code(e.g., P0300). "
        "Our assistant shall return a  structured diagnosis and safety guidance.Please Follow the Guidelines Carefully"

    )

    with gr.Row():
        symptoms_in = gr.Textbox(
            lines=4,
            label="Symptoms ",
            placeholder="e.g., Clicking sound when starting, no warning lights"
        )
        obd_in = gr.Textbox(
            lines=1,
            label="OBD-II Code",
            placeholder="e.g., P0300, P0171"
        )

    diagnose_btn = gr.Button("Diagnose ✅")
    output_box = gr.Textbox(lines=16, label="Diagnosis & Guidance ")

    diagnose_btn.click(fn=diagnose, inputs=[symptoms_in, obd_in], outputs=output_box)

    gr.Markdown("## Quick Examples (for demo)")
    gr.Examples(
        examples=EXAMPLES,
        inputs=[symptoms_in, obd_in]
    )

    with gr.Row():
        clear_btn = gr.Button("Clear")
        clear_btn.click(lambda: ("", ""), outputs=[symptoms_in, obd_in])

demo.launch(share=True, debug=False)


In [ ]:
import gradio as gr
import pandas as pd
import os

# =========================================================
# 0) Paths in Google Drive
# =========================================================
OBD_CSV_PATH = "/content/drive/MyDrive/mini/content/obd-trouble-codes.csv"
ICONS_DIR = "/content/drive/MyDrive/mini/content/icons"

# =========================================================
# 1) Login
# =========================================================
VALID_USERS = {
    "admin": "car123"
}

def do_login(username, password):
    u = (username or "").strip()
    p = (password or "").strip()
    if u in VALID_USERS and VALID_USERS[u] == p:
        return (
            gr.update(visible=False),   # hide login page
            gr.update(visible=True),    # show main app
            "",                         # clear login message
            u                           # save logged in user
        )
    return (
        gr.update(visible=True),
        gr.update(visible=False),
        "❌ Invalid username or password.",
        ""
    )

def do_logout():
    return (
        gr.update(visible=True),    # show login page
        gr.update(visible=False),   # hide main app
        "",                         # clear header
        ""                          # clear user state
    )

def render_header(u):
    if not u:
        return ""
    return f"""
    <div class="topbar">
        <div class="brand">🚗 Car Assistant LLM — Dashboard</div>
        <div class="userchip">✅ Logged in as: <b>{u}</b></div>
    </div>
    """

# =========================================================
# 2) Diagnose wrapper
# =========================================================
def safe_diagnose(symptoms, obd_text):
    if "diagnose" not in globals():
        return "❌ diagnose() function not found. Paste your Step-7 diagnose() code above this UI block."
    return diagnose(symptoms, obd_text)

# =========================================================
# 3) OBD DB loader
# =========================================================
def load_obd_db(csv_path: str):
    if not os.path.exists(csv_path):
        return {"P0300": "Random/Multiple Cylinder Misfire Detected"}

    df = pd.read_csv(csv_path)

    cols = {c.lower().strip(): c for c in df.columns}

    code_candidates = ["code", "dtc", "obd", "trouble_code"]
    desc_candidates = ["description", "meaning", "definition", "fault", "details"]

    code_col = None
    desc_col = None

    for k in code_candidates:
        if k in cols:
            code_col = cols[k]
            break

    for k in desc_candidates:
        if k in cols:
            desc_col = cols[k]
            break

    if code_col is None or desc_col is None:
        code_col = df.columns[0]
        desc_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]

    db = {}
    for _, row in df.iterrows():
        code = str(row[code_col]).strip().upper()
        desc = str(row[desc_col]).strip()
        if code and code != "NAN":
            db[code] = desc

    return db

def obd_list_text(db):
    return "\n".join(sorted(db.keys()))

def obd_lookup(code, db):
    c = (code or "").strip().upper()
    if not c:
        return "Enter an OBD code like P0300."
    return f"{c}: {db.get(c, 'Not found in loaded OBD list.')}"

def obd_search(query, db):
    q = (query or "").strip().upper()
    if not q:
        return obd_list_text(db)
    matches = []
    for k, v in db.items():
        if q in k or q in str(v).upper():
            matches.append(k)
    return "\n".join(sorted(matches)) if matches else "No matches found."

# Auto-load OBD database from Drive
OBD_DB = load_obd_db(OBD_CSV_PATH)
# =========================================================
# 4) Warning Light Gallery — FIXED
# =========================================================
import tempfile, textwrap

WARNING_LIGHTS = [
    ("engine_light.png", "Check Engine (MIL)",  "Engine/emissions fault. If flashing: avoid driving; scan OBD codes immediately.", "🔧", "#f59e0b"),
    ("oil_pressure.png",          "Oil Pressure",         "STOP engine ASAP. Low oil pressure can cause severe engine damage.",              "🛢",  "#ef4444"),
    ("battery_changing.png",      "Battery / Charging",   "Charging system fault. Vehicle may stall. Check alternator and battery.",         "🔋", "#3b82f6"),
    ("coolant_temp.png",         "Coolant Temperature",  "Overheating risk. Stop safely, let engine cool, and check coolant when safe.",    "🌡",  "#ef4444"),
    ("abs.png",          "ABS",                  "ABS may be disabled. Normal brakes work, but reduced control; service soon.",     "🛑", "#f97316"),
    ("airbag.png",       "Airbag / SRS",         "Airbag system fault. Safety risk; service recommended.",                         "💥", "#8b5cf6"),
]

_SVG_CACHE = {}   # avoid re-creating temp files on every call

def _make_svg(emoji: str, color: str, title: str) -> str:
    """Generate a coloured SVG placeholder and cache the temp file path."""
    key = (emoji, color, title)
    if key in _SVG_CACHE:
        return _SVG_CACHE[key]
    svg = textwrap.dedent(f"""\
        <svg xmlns="http://www.w3.org/2000/svg" width="160" height="160" viewBox="0 0 160 160">
          <rect width="160" height="160" rx="20" fill="{color}" opacity="0.12"/>
          <rect x="3" y="3" width="154" height="154" rx="18"
                fill="none" stroke="{color}" stroke-width="3"/>
          <text x="80" y="82" font-size="68" text-anchor="middle"
                dominant-baseline="middle">{emoji}</text>
          <text x="80" y="140" font-size="12" text-anchor="middle"
                fill="{color}" font-family="sans-serif"
                font-weight="bold">{title[:16]}</text>
        </svg>""")
    tmp = tempfile.NamedTemporaryFile(suffix=".svg", delete=False)
    tmp.write(svg.encode())
    tmp.close()
    _SVG_CACHE[key] = tmp.name
    return tmp.name

def build_warning_items():
    """Always returns all 6 items — real PNG or SVG placeholder."""
    items = []
    for fname, title, _desc, emoji, color in WARNING_LIGHTS:
        real = os.path.join(ICONS_DIR, fname)
        path = real if os.path.exists(real) else _make_svg(emoji, color, title)
        items.append((path, title))   # (image_path, caption)
    return items                      # guaranteed length == 6

def warning_explain(evt: gr.SelectData):
    """Index always maps directly to WARNING_LIGHTS — no filtering."""
    idx = evt.index
    if idx is None or idx >= len(WARNING_LIGHTS):
        return "Select an icon to see details."
    _, title, desc, _emoji, _color = WARNING_LIGHTS[idx]
    return f"### {title}\n\n{desc}"

# 5) Common Problems
# =========================================================
COMMON_PROBLEMS = [
    ("Clicking sound on start", "Weak battery / loose terminals / starter relay issue"),
    ("Engine overheating", "Low coolant / radiator fan / thermostat / water pump issue"),
    ("Grinding brakes", "Worn brake pads/rotors — inspect urgently"),
    ("Vibration at high speed", "Wheel balancing / alignment / worn tires"),
    ("Poor mileage", "Low tire pressure / dirty air filter / O2 sensor / driving style"),
]

def common_text():
    return "\n\n".join([f"• {title}\n  - {desc}" for title, desc in COMMON_PROBLEMS])

# =========================================================
# 6) Navigation helper
# =========================================================
def show_page(page):
    return (
        gr.update(visible=(page == "assistant")),
        gr.update(visible=(page == "obd")),
        gr.update(visible=(page == "lights")),
        gr.update(visible=(page == "common")),
    )

# =========================================================
# 7) Styling
# =========================================================
CSS = """
.topbar {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 14px 16px;
    border-bottom: 1px solid #e6e6e6;
    background: #ffffff;
}
.brand {
    font-size: 22px;
    font-weight: 800;
}
.userchip {
    font-size: 13px;
    padding: 8px 12px;
    border: 1px solid #e6e6e6;
    border-radius: 999px;
    background: #fafafa;
}
.sidebar-btn button {
    width: 100% !important;
    justify-content: flex-start !important;
    font-weight: 700 !important;
}
#gallery_full {
    max-height: 620px !important;
    overflow-y: auto !important;
}
#gallery_full .grid-wrap {
    gap: 12px !important;
}
#gallery_full img {
    border-radius: 12px !important;
    object-fit: contain !important;
    max-height: 180px !important;
}
"""

theme = gr.themes.Soft()

# =========================================================
# 8) Build UI
# =========================================================
with gr.Blocks(title="Car Assistant LLM — Dashboard") as app:
    state_user = gr.State("")
    state_obd_db = gr.State(OBD_DB)

    # ---------------- LOGIN PAGE ----------------
    login_page = gr.Group(visible=True)
    with login_page:
        gr.Markdown("## 🔐 Login")
        in_user = gr.Textbox(label="Username", placeholder="admin")
        in_pass = gr.Textbox(label="Password", type="password", placeholder="car123")
        btn_login = gr.Button("Login")
        login_msg = gr.Markdown("")

    # ---------------- MAIN APP ----------------
    main_app = gr.Group(visible=False)
    with main_app:
        header = gr.HTML(render_header(""))

        with gr.Row():
            # -------- SIDEBAR --------
            with gr.Column(scale=1, min_width=240):
                btn_assistant = gr.Button("🧰  Car Assistant", elem_classes=["sidebar-btn"])
                btn_obd = gr.Button("🔎  OBD Code Database", elem_classes=["sidebar-btn"])
                btn_lights = gr.Button("🚨  Warning Light Icons", elem_classes=["sidebar-btn"])
                btn_common = gr.Button("🛠  Common Problems", elem_classes=["sidebar-btn"])
                gr.Markdown("---")
                btn_logout = gr.Button("🚪 Logout")

            # -------- MAIN CONTENT --------
            with gr.Column(scale=4):
                # Assistant page
                page_assistant = gr.Group(visible=True)
                with page_assistant:
                    gr.Markdown("### 🧰 Car Assistant (Symptoms → Guidance)")
                    symptoms = gr.Textbox(
                        lines=5,
                        label="Symptoms",
                        placeholder="e.g., Clicking sound when starting, no warning lights"
                    )
                    obd_codes = gr.Textbox(
                        lines=1,
                        label="OBD Codes (optional)",
                        placeholder="e.g., P0300, P0171"
                    )
                    btn_diag = gr.Button("Diagnose ✅")
                    out_diag = gr.Textbox(lines=16, label="Diagnosis & Guidance")
                    btn_diag.click(fn=safe_diagnose, inputs=[symptoms, obd_codes], outputs=out_diag)

                # OBD page
                page_obd = gr.Group(visible=False)
                with page_obd:
                    gr.Markdown("### 🔎 OBD Code Database (Auto-loaded from Drive)")
                    gr.Markdown(f"Loaded file: `{OBD_CSV_PATH}`  \nTotal codes: **{len(OBD_DB)}**")

                    with gr.Row():
                        search_q = gr.Textbox(label="Search (code/keyword)", placeholder="misfire / P03")
                        btn_search = gr.Button("Search")

                    codes_list = gr.Textbox(
                        lines=14,
                        value=obd_list_text(OBD_DB),
                        label="Available Codes (filtered)"
                    )

                    with gr.Row():
                        lookup_code = gr.Textbox(label="Lookup code", placeholder="P0302")
                        btn_lookup = gr.Button("Lookup")

                    meaning = gr.Textbox(lines=6, label="Meaning")

                    btn_search.click(
                        fn=lambda q, db: obd_search(q, db),
                        inputs=[search_q, state_obd_db],
                        outputs=codes_list
                    )
                    btn_lookup.click(
                        fn=lambda c, db: obd_lookup(c, db),
                        inputs=[lookup_code, state_obd_db],
                        outputs=meaning
                    )
                # Warning lights page
                page_lights = gr.Group(visible=False)
                with page_lights:
                    gr.Markdown("### 🚨 Warning Light Icons")
                    gr.Markdown(
                        f"Icons loaded from `{ICONS_DIR}`.  \n"
                        f"Missing icons show coloured placeholders automatically."
                    )
                    # Always 6 items — no conditional needed
                    gallery = gr.Gallery(
                        label="Warning Lights (click for details)",
                        value=build_warning_items(),
                        columns=3,
                        rows=2,
                        height=440,
                        object_fit="contain",
                        elem_id="gallery_full"
                    )
                    explain_box = gr.Markdown("👆 Select an icon above to see details.")
                    gallery.select(fn=warning_explain, inputs=None, outputs=explain_box)

                # Common problems page
                page_common = gr.Group(visible=False)
                with page_common:
                    gr.Markdown("### 🛠 Common Car Problems")
                    common_box = gr.Textbox(lines=18, value=common_text(), label="Quick Reference")

        # Sidebar navigation
        btn_assistant.click(
            fn=lambda: show_page("assistant"),
            inputs=None,
            outputs=[page_assistant, page_obd, page_lights, page_common]
        )
        btn_obd.click(
            fn=lambda: show_page("obd"),
            inputs=None,
            outputs=[page_assistant, page_obd, page_lights, page_common]
        )
        btn_lights.click(
            fn=lambda: show_page("lights"),
            inputs=None,
            outputs=[page_assistant, page_obd, page_lights, page_common]
        )
        btn_common.click(
            fn=lambda: show_page("common"),
            inputs=None,
            outputs=[page_assistant, page_obd, page_lights, page_common]
        )

        # Logout
        btn_logout.click(
            fn=do_logout,
            inputs=None,
            outputs=[login_page, main_app, header, state_user]
        )

    # Login wiring
    btn_login.click(
        fn=do_login,
        inputs=[in_user, in_pass],
        outputs=[login_page, main_app, login_msg, state_user]
    ).then(
        fn=render_header,
        inputs=[state_user],
        outputs=[header]
    )

app.launch(share=True, theme=theme, css=CSS)